# 01 — Bona Data Analysis
## SunnyBest Retail Forecasting System — Dataset Exploration

This notebook analyses all available datasets in the SunnyBest SFS project:
- `sunnybest_merged_df.csv` — Master merged dataset (daily grain)
- `weekly_sales_v4_promotions.csv` — Latest weekly model input (v4)
- `weekly_sales_v3_calendar.csv` — Weekly v3 with calendar features
- `weekly_sales_v2.csv` — Weekly v2 baseline
- `weekly_sales.csv` — Weekly v1 (first iteration)
- `elasticity_by_category.csv` — Price elasticity per category
- `weekly_forecasts.csv` — Model predictions
- `weekly_actuals.csv` — Ground truth for monitoring

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BASE = "../data/processed"
OUT  = "../data/outputs"

# Load all datasets
df       = pd.read_csv(f"{BASE}/sunnybest_merged_df.csv", parse_dates=["date"])
wk_v4    = pd.read_csv(f"{BASE}/weekly_sales_v4_promotions.csv")
wk_v3    = pd.read_csv(f"{BASE}/weekly_sales_v3_calendar.csv")
wk_v2    = pd.read_csv(f"{BASE}/weekly_sales_v2.csv")
wk_v1    = pd.read_csv(f"{BASE}/weekly_sales.csv")
elast    = pd.read_csv(f"{BASE}/elasticity_by_category.csv")
forecasts = pd.read_csv(f"{OUT}/weekly_forecasts.csv", parse_dates=["week_start"])
actuals   = pd.read_csv(f"{OUT}/weekly_actuals.csv", parse_dates=["week_start"])

print("All datasets loaded successfully.")

---
## 1. Dataset Overview — Shape, Columns & Missing Values

In [ ]:
datasets = {
    "merged_df (daily)":        df,
    "weekly_v4_promotions":     wk_v4,
    "weekly_v3_calendar":       wk_v3,
    "weekly_v2":                wk_v2,
    "weekly_v1":                wk_v1,
    "elasticity_by_category":   elast,
    "weekly_forecasts":         forecasts,
    "weekly_actuals":           actuals,
}

summary_rows = []
for name, d in datasets.items():
    missing_pct = (d.isnull().sum().sum() / d.size * 100).round(2)
    summary_rows.append({
        "Dataset":       name,
        "Rows":          f"{len(d):,}",
        "Columns":       d.shape[1],
        "Missing %":     f"{missing_pct}%",
        "Date Range":    f"{d.iloc[:,0].min()} → {d.iloc[:,0].max()}" if "date" in d.columns or "week_start" in d.columns else "—",
    })

pd.DataFrame(summary_rows).set_index("Dataset")

In [ ]:
# Column-level missing values for the master dataset
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing values in the master merged dataset.")
else:
    print("Columns with missing values:")
    display(missing.to_frame("missing_count").assign(pct=lambda x: (x/len(df)*100).round(2)))

---
## 2. Master Dataset — Key Statistics & Distributions

In [ ]:
print(f"Date range : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Stores     : {df['store_id'].nunique()} unique stores")
print(f"Products   : {df['product_id'].nunique()} unique products")
print(f"Categories : {df['category'].nunique()} — {list(df['category'].unique())}")
print(f"Cities     : {df['city'].nunique()} — {list(df['city'].unique())}")
print(f"Total rows : {len(df):,}")
print()
df[["units_sold", "revenue", "price", "discount_pct", "starting_inventory"]].describe().round(2)